# 05 Feedback loop: learning curve over the ratio of derived items

Items predicted by notebook 04 and then written back as categories come back in the labelled pull.
This notebook measures how adding them to training changes accuracy, **as a function of how many are
added**, and extrapolates to the ratio used in production (derived rows / organic rows).

How each point is measured, so nothing leaks:

1. Organic rows (labels not derived from predictions) are split into `N_FOLDS` ID-hash folds, as in
   notebook 02. Fold *k* is the test set; nothing below ever trains on it.
2. For each organic fraction *g*, a nested hash subsample of the other folds is the organic training set.
   The **teacher** is trained on it, exactly as production trains on all organic rows.
3. The teacher re-labels the pool (every item the model predicted) and the acceptance rule is re-applied.
   The production labels of derived rows are never used: that model has seen fold *k*.
4. Students are trained on the organic rows plus the first `ratio x organic` accepted items, and scored
   on fold *k*. Ratio 0 is the teacher itself.

The accuracy gain is fitted as a smooth function of ratio and organic size and read off at the production
ratio, with a paired bootstrap over test rows for the uncertainty.

Every cell is saved to `outputs/feedback_learning_curve_<timestamp>/` when it finishes. If the session
dies, set `RESUME_FROM` to that folder and run all cells again.

In [ ]:
# ---- MLflow -------------------------------------------------------------------
USE_MLFLOW = False                    # False = no tracking at all
MLFLOW_EXPERIMENT = None              # None = this notebook's experiment; or e.g. "/Users/patrick.white@vallen.com/nb-classifier"

# ---- experiment -----------------------------------------------------------------
N_FOLDS = 3                           # ID-hash folds of the organic rows (test folds)
FOLD_SEED = 0
FOLDS_TO_RUN = None                   # None = all; e.g. [0] for a quick first look
ORGANIC_FRACTIONS = [0.25, 0.5, 1.0]  # shares of the training folds' organic rows (nested subsamples)
RATIOS = [0.1, 0.25, 0.5, 1.0, 2.0, 4.0]  # derived rows / organic rows added to each student; capped at
                                      # what the teacher accepts (reported as 'capped')
SAMPLE_SEED = 0                       # seed of the nested subsamples
PRODUCTION_RATIO = None               # None = derived rows / organic rows in the data
DEPTH = None                          # level the analysis uses (None = deepest, i.e. the full path)
N_BOOT = 300                          # paired bootstrap replicates over test rows
CI = 0.90                             # width of the intervals
RESUME_FROM = None                    # e.g. "feedback_learning_curve_20261020_143000"

In [ ]:
# ---- prepared data ------------------------------------------------------------
PREPARED_DATA = None                  # a data/<timestamp> folder from notebook 01 ("latest" = newest);
                                      # None = pull and clean below with the settings in this cell

# ---- data source (RAW_DATA_FILE supersedes SQL_QUERY) -----------------------
RAW_DATA_FILE = None                  # e.g. "data/my_extract.txt"; read with dtype=str
RAW_DATA_DELIMITER = "|"              # delimiter of RAW_DATA_FILE
RAW_DATA_QUOTED = True                # False if RAW_DATA_FILE uses no quoting and descriptions may start with "
SQL_QUERY = "labeled_items"           # a file in data/queries/ (name without .sql); may be None only if RAW_DATA_FILE is set

# ---- items to exclude ---------------------------------------------------------
BLACKLIST_FILE = None                 # e.g. "data/blacklists/blacklist.txt" (one column)
BLACKLIST_COL = "Vallen ID"           # data column the blacklist values are matched against
BLACKLIST_HAS_HEADER = True           # first line of the blacklist file is a header

# ---- cleaning -----------------------------------------------------------------
CLEANING_METHOD = "cleaning_method_20261015"   # a folder in cleaning/ (code, term lists, settings in method.yaml)
APPEND_FIELDS = {}                    # e.g. {"Global Vendno": "VEND", "Prodline": "PLINE"} -> VEND_<value> tokens
APPEND_NULLS_AS_NONE = True           # True: null field -> VEND_None;  False: no marker for that row

# ---- columns ------------------------------------------------------------------
TEXT_COL = "FullDesc"                 # raw description field to clean
CLEANED_COL = "CleanDesc"             # header for the cleaned description
LEVEL_COLUMNS = ["Level 1 ID", "Level 2 ID", "Level 3 ID", "Level 4 ID"]      # labels the model learns
ID_COL = "ID"                         # item key: matched against the prediction files, hashed for folds and subsamples

# ---- which labelled rows came from predictions ----------------------------------
DERIVED_IDS_FILE = "data/derived_ids.txt"  # one-column .txt/.csv of the IDs of items already predicted for
                                      # (same format as the blacklist files); labelled rows with these IDs
                                      # are "derived", every other labelled row is "organic"
DERIVED_IDS_HAS_HEADER = True         # first line of DERIVED_IDS_FILE is a header
PREDICTION_FILES = None               # optional: notebook-04 outputs the derived labels came from ("all" = every
                                      # outputs/predictions_*, a folder name, a CSV path, or a list). Adds the
                                      # predicted-but-not-accepted items to the pool and the production
                                      # confidences to the diagnostics. Without DERIVED_IDS_FILE, a labelled
                                      # row whose ID is in these files counts as derived.
DERIVED_FLAG_COL = None               # alternative to the ID file: a data column marking derived rows (1/true/yes)
ACCEPT_LEVEL = 4                      # the fold teachers accept a pool item when this level's confidence ...
ACCEPT_MIN_CONFIDENCE = None          # ... is at least this; None = accept every item. With only DERIVED_IDS_FILE the
                                      # pool is exactly the items that were accepted, so leave this None.
                                      # Set it only when PREDICTION_FILES adds the items that were not accepted.

In [ ]:
# ==== HYPERPARAMETERS ==========================================================
# Single values only: the production model's settings (as in notebook 03). Every teacher and
# student in this notebook is trained with them.

HYPERPARAMETERS = {
    "vectorizer_kind":     "tfidf",
    "max_features":        None,
    "ngram_range":         (1, 2),
    "min_df":              2,
    "max_df":              0.95,
    "stop_words":          "english",
    "sublinear_tf":        True,
    "binary":              True,
    "norm":                "l2",
    "use_idf":             False,
    "ordered_bigrams":     True,
    "token_pattern":       "whitespace",
    "alpha":               0.001,
    "fit_prior":           True,
    "prediction_mode":     "conditional",
    "rare_path_threshold": 5,
}

In [ ]:
# ---- setup (no settings below this line need changing) ----------------------
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from nbclassifier import config, data, feedback, paths
from nbclassifier.tracking import Tracker

KIND = "learning_curve"
hp = config.expand_grid(HYPERPARAMETERS, grid_search=False)[0]
rule = feedback.SelectionRule(level=ACCEPT_LEVEL, min_confidence=ACCEPT_MIN_CONFIDENCE)
CONFIG = {
    "hyperparameters": hp.to_dict(), "n_folds": N_FOLDS, "fold_seed": FOLD_SEED, "sample_seed": SAMPLE_SEED,
    "rule": rule.to_dict(),
}
tracker = Tracker(USE_MLFLOW, MLFLOW_EXPERIMENT, run_name=f"05_feedback_learning_curve_{paths.timestamp()}",
                  tags={"notebook": "05_feedback_learning_curve"})

## Data

Labelled rows whose ID is listed in `DERIVED_IDS_FILE` are *derived*; all others are *organic*. The pool
the fold teachers re-label is the derived rows (plus the predicted-but-not-accepted items, if
`PREDICTION_FILES` is set). Check the diagnostics:

* **IDs found in the labelled data**: listed IDs missing from the query results are left out. Many
  missing usually means a different ID format or a blacklist.
* **recorded label = production prediction** (only with `PREDICTION_FILES`): 1.0 means the derived
  labels are pure model output; less means some were corrected after prediction. The experiments never
  use the recorded labels: the fold teachers re-label the pool, because the production model has seen
  every test fold. That is exact when the recorded labels are the model's predictions; the more were
  corrected, the more these experiments understate the real derived labels' quality.
* **production confidence of accepted vs not-accepted pool items** (only with `PREDICTION_FILES`): if
  the ranges don't overlap, the minimum of the accepted ones is the threshold that was in use; set
  `ACCEPT_MIN_CONFIDENCE` to it.

In [ ]:
if RESUME_FROM:
    store = feedback.Store.open(RESUME_FROM, KIND, CONFIG)
    fd = store.data()
    cols = feedback.Cols(**store.config["columns"])
    print(f"Resuming {store.folder.name}: the data saved in that folder is used; data settings above are ignored.")
else:
    if PREPARED_DATA:
        prepared = data.load_dataset(PREPARED_DATA)
        print(f"loaded {prepared.metadata['loaded_from']} (cleaning method {prepared.cleaner.method_name!r})")
        prepared = data.blacklist_prepared(prepared, BLACKLIST_FILE, BLACKLIST_COL, BLACKLIST_HAS_HEADER)
        TEXT_COL, CLEANED_COL = prepared.metadata["text_col"], prepared.metadata["cleaned_col"]
    else:
        prepared = data.prepare_dataset(
            raw_data_file=RAW_DATA_FILE, sql_query=SQL_QUERY, delimiter=RAW_DATA_DELIMITER,
            blacklist_file=BLACKLIST_FILE, blacklist_col=BLACKLIST_COL, blacklist_has_header=BLACKLIST_HAS_HEADER,
            cleaning_method=CLEANING_METHOD,
            append_fields=APPEND_FIELDS, append_nulls_as_none=APPEND_NULLS_AS_NONE,
            text_col=TEXT_COL, cleaned_col=CLEANED_COL, level_columns=LEVEL_COLUMNS,
            raw_data_quoted=RAW_DATA_QUOTED, spark=globals().get("spark"),
        )
    fd = feedback.split_feedback(prepared, level_columns=LEVEL_COLUMNS, text_col=TEXT_COL, cleaned_col=CLEANED_COL,
                                 id_col=ID_COL, derived_ids_file=DERIVED_IDS_FILE,
                                 derived_ids_has_header=DERIVED_IDS_HAS_HEADER, prediction_files=PREDICTION_FILES,
                                 derived_flag_col=DERIVED_FLAG_COL)
    del prepared
    cols = feedback.Cols(id=ID_COL, cleaned=CLEANED_COL, levels=list(LEVEL_COLUMNS))
    store = feedback.Store.create(KIND, {**CONFIG, "columns": cols.__dict__}, fd)
LEVEL_COLUMNS = cols.levels
DEPTH = DEPTH or len(LEVEL_COLUMNS)
N_PROD = len(fd.organic)                       # organic rows the production model is trained on
R_PROD = PRODUCTION_RATIO if PRODUCTION_RATIO is not None else fd.production_ratio
print(f"\nproduction: {N_PROD:,} organic rows, ratio {R_PROD:.3f}; analysis at L{DEPTH}")
print("evaluation folder:", store.folder)
feedback.describe(fd, LEVEL_COLUMNS, rule)

## Run the grid

One teacher per (fold, organic fraction) and one student per ratio. The log shows the realized ratio:
a ratio is **capped** when the teacher accepts fewer pool items than it needs. At a smaller organic
fraction the same items give a higher ratio, so a grid with several fractions reaches further.

In [ ]:
results = feedback.run_ratio_grid(
    store, fd, hp, cols, n_folds=N_FOLDS, fold_seed=FOLD_SEED, folds_to_run=FOLDS_TO_RUN,
    organic_fracs=ORGANIC_FRACTIONS, ratios=RATIOS, rule=rule, sample_seed=SAMPLE_SEED,
)

## Gain over ratio 0, per cell

`delta` is the student's accuracy minus the teacher's (ratio 0, same organic rows), pooled over the
folds that finished every cell. The interval is a paired bootstrap over test rows: both models are
scored on the same rows, so most of the row-to-row noise cancels.

In [ ]:
meta = store.meta()
meta = meta[meta["organic_frac"].isin(ORGANIC_FRACTIONS) & meta["ratio_target"].isin([0.0] + list(RATIOS))]
meta["cell"] = list(zip(meta["organic_frac"], meta["ratio_target"]))
n_cells = meta["cell"].nunique()
complete = meta.groupby("fold")["cell"].nunique()
folds_done = sorted(complete[complete == n_cells].index)
if not folds_done:
    raise RuntimeError("No fold has finished every cell yet")
meta = meta[meta["fold"].isin(folds_done)]
print(f"folds used: {folds_done}")

cells = sorted(meta["cell"].unique())
keys = [{int(r.fold): r.key for r in meta[meta["cell"] == c].itertuples()} for c in cells]
point, boot = feedback.bootstrap_cells(store, keys, DEPTH, n_boot=N_BOOT)
base_idx = [cells.index((g, 0.0)) for g, _ in cells]
d_point = point - point[base_idx]
d_boot = boot - boot[:, base_idx]
q = [(1 - CI) / 2, 1 - (1 - CI) / 2]

agg = meta.groupby("cell").agg(n_organic=("n_organic", "mean"), n_derived=("n_derived", "mean"),
                               ratio=("ratio", "mean"), capped=("capped", "any"))
table = pd.DataFrame({
    "organic_frac": [c[0] for c in cells], "ratio_target": [c[1] for c in cells],
    "ratio_realized": agg.loc[cells, "ratio"].to_numpy(), "capped": agg.loc[cells, "capped"].to_numpy(),
    "n_organic": agg.loc[cells, "n_organic"].round().astype(int).to_numpy(),
    "n_derived": agg.loc[cells, "n_derived"].round().astype(int).to_numpy(),
    f"L{DEPTH} accuracy": point,
    "delta": d_point, "delta_low": np.quantile(d_boot, q[0], axis=0), "delta_high": np.quantile(d_boot, q[1], axis=0),
    "delta_se": d_boot.std(axis=0, ddof=1),
})
table.round(4)

## Fitted surface and extrapolation to production

The gain is modelled as

    delta(r, n) = b1*z + b2*z^2 + b3*z*log(n / n_max),   z = log(1 + r)

where *r* is the realized ratio and *n* the organic rows. There is no intercept, since ratio 0 is the
reference. The fit is weighted by 1 / (bootstrap variance) and repeated on every bootstrap replicate,
which gives the interval at the production point (`R_PROD`, `N_PROD`).

The production point is *interpolated* if a tested cell reaches its ratio, and *extrapolated*
otherwise. The holdout check below tells you how far to trust extrapolation in this data.

In [ ]:
# capped ratios that ended with the same training rows are one point, not several
fit_rows = (table["ratio_target"] > 0) & ~table.duplicated(["organic_frac", "n_derived"])
X_r = table.loc[fit_rows, "ratio_realized"].to_numpy()
X_n = table.loc[fit_rows, "n_organic"].to_numpy()
w = 1.0 / np.maximum(table.loc[fit_rows, "delta_se"].to_numpy(), 1e-6) ** 2
n_ref = X_n.max()
surface = feedback.RatioSurface(n_ref).fit(X_r, X_n, table.loc[fit_rows, "delta"].to_numpy(), w)
surface_boot = feedback.RatioSurface(n_ref).fit(X_r, X_n, d_boot[:, np.flatnonzero(fit_rows)].T, w)

pred_point = float(surface.predict(R_PROD, N_PROD)[0, 0])
pred_boot = surface_boot.predict(R_PROD, N_PROD).ravel()
lo, hi = np.quantile(pred_boot, q)
max_r = table.loc[fit_rows, "ratio_realized"].max()
max_r_full = table.loc[fit_rows & (table["organic_frac"] == max(ORGANIC_FRACTIONS)), "ratio_realized"].max()
where = ("interpolated" if R_PROD <= max_r else f"EXTRAPOLATED beyond the largest tested ratio ({max_r:.2f})")

print(f"production point: ratio {R_PROD:.3f}, {N_PROD:,} organic rows ({where})")
print(f"  largest ratio reached with all organic rows: {max_r_full:.3f}")
print(f"predicted change in L{DEPTH} accuracy from including the derived items: "
      f"{pred_point:+.4f}  ({CI:.0%} interval {lo:+.4f} to {hi:+.4f})")
if hi < 0:
    print("=> including them is expected to LOWER accuracy")
elif lo > 0:
    print("=> including them is expected to RAISE accuracy")
else:
    print("=> the interval includes zero: no reliable gain or loss at this ratio")
tracker.log_metrics({"predicted_delta": pred_point, "predicted_delta_low": lo, "predicted_delta_high": hi,
                     "production_ratio": R_PROD})

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5.5))
colors = plt.cm.viridis(np.linspace(0.15, 0.85, len(ORGANIC_FRACTIONS)))
for g, col in zip(sorted(ORGANIC_FRACTIONS), colors):
    t = table[table["organic_frac"] == g].sort_values("ratio_realized")
    ax.errorbar(t["ratio_realized"], t["delta"], yerr=[t["delta"] - t["delta_low"], t["delta_high"] - t["delta"]],
                fmt="o", color=col, capsize=3, label=f"organic fraction {g} (n ~ {int(t['n_organic'].iloc[0]):,})")
    rr = np.linspace(0, t["ratio_realized"].max(), 100)       # each curve only over its tested range
    ax.plot(rr, surface.predict(rr, np.full_like(rr, t["n_organic"].iloc[0])).ravel(), color=col, lw=1, alpha=0.8)
rr = np.linspace(0, max(max_r, R_PROD) * 1.05, 100)
ax.plot(rr, surface.predict(rr, np.full_like(rr, N_PROD)).ravel(), color="black", ls="--", lw=1.2,
        label=f"fit at production size ({N_PROD:,})")
ax.axvline(R_PROD, color="firebrick", ls=":", lw=1.2)
ax.errorbar([R_PROD], [pred_point], yerr=[[pred_point - lo], [hi - pred_point]], fmt="D", color="firebrick",
            capsize=4, label=f"production ratio {R_PROD:.2f}: {pred_point:+.4f}")
ax.axhline(0, color="gray", lw=0.8)
ax.set_xlabel("derived rows / organic rows in training (realized)")
ax.set_ylabel(f"change in L{DEPTH} accuracy vs ratio 0")
ax.set_title("Gain from adding teacher-labelled pool items (clean test folds)")
ax.legend(fontsize=8)
fig.tight_layout()
plot_dir = store.folder / "plots"
plot_dir.mkdir(exist_ok=True)
fig.savefig(plot_dir / f"learning_curve_L{DEPTH}.png", dpi=120, bbox_inches="tight")
tracker.log_figure(fig, f"plots/learning_curve_L{DEPTH}.png")

## How well does the surface extrapolate here?

For each organic fraction, the cell with the largest ratio is left out, the surface is refitted on the
rest, and the left-out cell is predicted. Errors that are large compared with the intervals mean the
extrapolated production estimate should not be trusted beyond the tested range.

In [ ]:
rows = []
for g in sorted(ORGANIC_FRACTIONS):
    in_g = fit_rows & (table["organic_frac"] == g)
    if in_g.sum() < 2:
        continue
    out_i = table.loc[in_g, "ratio_realized"].idxmax()
    keep = fit_rows & (table.index != out_i)
    s = feedback.RatioSurface(n_ref).fit(table.loc[keep, "ratio_realized"], table.loc[keep, "n_organic"],
                                         table.loc[keep, "delta"], 1.0 / np.maximum(table.loc[keep, "delta_se"], 1e-6) ** 2)
    p = float(s.predict(table.loc[out_i, "ratio_realized"], table.loc[out_i, "n_organic"])[0, 0])
    rows.append({"organic_frac": g, "left_out_ratio": table.loc[out_i, "ratio_realized"],
                 "observed_delta": table.loc[out_i, "delta"], "observed_low": table.loc[out_i, "delta_low"],
                 "observed_high": table.loc[out_i, "delta_high"], "predicted_delta": p,
                 "inside_interval": table.loc[out_i, "delta_low"] <= p <= table.loc[out_i, "delta_high"]})
holdout = pd.DataFrame(rows)
holdout.round(4)

## By level, at the tested cell nearest the production ratio (all organic rows)

In [ ]:
full = table[(table["organic_frac"] == max(ORGANIC_FRACTIONS)) & (table["ratio_target"] > 0)]
near = full.loc[(full["ratio_realized"] - R_PROD).abs().idxmin()]
target = near["ratio_target"]
pair = [(max(ORGANIC_FRACTIONS), 0.0), (max(ORGANIC_FRACTIONS), target)]
pair_keys = [{int(r.fold): r.key for r in meta[meta["cell"] == c].itertuples()} for c in pair]
rows = []
for d in range(1, len(LEVEL_COLUMNS) + 1):
    p, b = feedback.bootstrap_cells(store, pair_keys, d, n_boot=N_BOOT, seed=d)
    db = b[:, 1] - b[:, 0]
    rows.append({"level": f"L{d}", "accuracy without": p[0], "accuracy with": p[1], "delta": p[1] - p[0],
                 "delta_low": np.quantile(db, q[0]), "delta_high": np.quantile(db, q[1])})
print(f"cell: ratio target {target} (realized {near['ratio_realized']:.3f}), organic fraction {max(ORGANIC_FRACTIONS)}")
pd.DataFrame(rows).set_index("level").round(4)

In [ ]:
table.to_csv(store.folder / f"cells_L{DEPTH}.csv", index=False)
tracker.log_table(table, f"cells_L{DEPTH}.csv")
tracker.end()